# 05 - Fine-tuned DeBERTa-v3-base

`AutoModelForMultipleChoice` sees all five (prompt, option) pairs per question and produces a softmax over the 5 options. Cross-entropy loss trains the correct-option index. 

## Setup

In [46]:
import subprocess, sys
for pkg in ['wandb', 'pyarrow', 'transformers', 'sentencepiece', 'accelerate']:
    try: __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

In [47]:
import os
from pathlib import Path

IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    DATA_DIR = Path('/kaggle/input/competitions/smart-mcq-solver-challenge')
    OUTPUT_DIR = Path('/kaggle/working')
else:
    ROOT = Path.cwd()
    if ROOT.name == 'notebooks': ROOT = ROOT.parent
    DATA_DIR = ROOT / 'data'
    OUTPUT_DIR = ROOT

PREDICTIONS_DIR = OUTPUT_DIR / 'predictions'
FEATURES_DIR = OUTPUT_DIR / 'features'
for p in [PREDICTIONS_DIR, FEATURES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

In [48]:
import os
for root, dirs, files in os.walk('/kaggle/input/datasets/krishnaanalyst/mcq-solver-krishna3'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/krishnaanalyst/mcq-solver-krishna3/train_meta.parquet
/kaggle/input/datasets/krishnaanalyst/mcq-solver-krishna3/test_long.parquet
/kaggle/input/datasets/krishnaanalyst/mcq-solver-krishna3/test_meta.parquet
/kaggle/input/datasets/krishnaanalyst/mcq-solver-krishna3/train_long.parquet
/kaggle/input/datasets/krishnaanalyst/mcq-solver-krishna3/feat_cols.json


In [49]:
FEATURES_DIR = Path('/kaggle/input/datasets/krishnaanalyst/mcq-solver-krishna3')

In [50]:
if IS_KAGGLE:
    for d in Path('/kaggle/input').iterdir():
        if d.is_dir() and (d / 'train_meta.parquet').exists():
            FEATURES_DIR = d
            break
    print(f'FEATURES_DIR: {FEATURES_DIR}')
    assert (FEATURES_DIR / 'train_meta.parquet').exists(), 'features not found'

FEATURES_DIR: /kaggle/input/datasets/krishnaanalyst/mcq-solver-krishna3


In [51]:
import wandb
WANDB_ENTITY = '23f2001849-dl-genai-project'
WANDB_PROJECT = '23f2001849-t22026'
USE_WANDB = False
try:
    if IS_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('WANDB_API_KEY')
    else:
        key = os.environ.get('WANDB_API_KEY', '').strip()
    if key:
        wandb.login(key=key, relogin=True)
        USE_WANDB = True
except Exception as e:
    print(f'W&B setup failed: {e}')
print(f'W&B enabled: {USE_WANDB}')

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


W&B enabled: True


In [52]:
import json, time, random, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForMultipleChoice,
                          get_linear_schedule_with_warmup)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
OPTIONS = ['A', 'B', 'C', 'D', 'E']
NUM_CHOICES = 5
FOLDS = 5
print(f'device: {device}')

device: cuda


## Config

In [53]:
MODEL_ID = 'microsoft/deberta-v3-base'
MAX_LEN = 192
BATCH = 4
ACCUM = 2
EPOCHS = 4
PATIENCE = 2
LR = 5e-6

## Load data

In [54]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
train_meta = pd.read_parquet(FEATURES_DIR / 'train_meta.parquet')
train['fold'] = train['id'].map(dict(zip(train_meta['id'], train_meta['fold'])))
print(f'train: {train.shape}')
print('fold sizes:', train.groupby('fold').size().tolist())

train: (2000, 9)
fold sizes: [400, 400, 400, 400, 400]


## Dataset

In [55]:
class MCDataset(Dataset):
    def __init__(self, df, tok, has_answer=True):
        self.df = df.reset_index(drop=True)
        self.tok = tok
        self.has_answer = has_answer

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        prompts = [row['prompt']] * NUM_CHOICES
        options = [str(row[L]) for L in OPTIONS]
        enc = self.tok(prompts, options, truncation=True, max_length=MAX_LEN,
                       padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.has_answer:
            item['labels'] = torch.tensor(OPTIONS.index(row['answer']), dtype=torch.long)
        return item

## Metric helpers

In [56]:
def map3_score(pred, true):
    s = 0.0
    for p, a in zip(pred, true):
        for k, L in enumerate(p[:3]):
            if L == a:
                s += 1.0 / (k + 1); break
    return s / len(true)

def scores_to_top3(grid):
    order = np.argsort(-grid, axis=1)
    return [[OPTIONS[i] for i in row[:3]] for row in order]

## Tokenizer

In [57]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print(f'tokenizer: {tokenizer.__class__.__name__}')

tokenizer: DebertaV2Tokenizer


## Training step

In [58]:
def train_one_epoch(model, loader, optim, sched, scaler):
    model.train()
    optim.zero_grad()
    total = 0.0
    for step, batch in enumerate(loader):
        labels = batch.pop('labels').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}
        if scaler is not None:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                out = model(**batch, labels=labels)
            scaler.scale(out.loss / ACCUM).backward()
            if (step + 1) % ACCUM == 0:
                scaler.step(optim); scaler.update()
                sched.step(); optim.zero_grad()
        else:
            out = model(**batch, labels=labels)
            (out.loss / ACCUM).backward()
            if (step + 1) % ACCUM == 0:
                optim.step(); sched.step(); optim.zero_grad()
        total += float(out.loss.detach().cpu())
    return total / max(len(loader), 1)

In [59]:
def eval_loader(model, loader):
    model.eval()
    probs, labels_all, loss_sum, n = [], [], 0.0, 0
    loss_fn = nn.CrossEntropyLoss()
    with torch.no_grad():
        for batch in loader:
            labels = batch.pop('labels').to(device)
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss_sum += float(loss_fn(out.logits, labels).item()) * labels.size(0)
            n += labels.size(0)
            probs.append(torch.softmax(out.logits, dim=-1).cpu().numpy())
            labels_all.append(labels.cpu().numpy())
    return np.concatenate(probs), np.concatenate(labels_all), loss_sum / max(n, 1)

## Train one fold

Best checkpoint by val MAP@3. Early stop when patience exhausted.

In [60]:
def train_fold(train_df, val_df, fold_idx):
    torch.manual_seed(SEED)
    model = AutoModelForMultipleChoice.from_pretrained(
        MODEL_ID, torch_dtype=torch.float32).to(device)

    tr_ds = MCDataset(train_df, tokenizer, has_answer=True)
    va_ds = MCDataset(val_df, tokenizer, has_answer=True)
    tr_dl = DataLoader(tr_ds, batch_size=BATCH, shuffle=True, num_workers=0)
    va_dl = DataLoader(va_ds, batch_size=BATCH, shuffle=False, num_workers=0)

    steps = max((len(tr_dl) // ACCUM) * EPOCHS, 1)
    optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(optim, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler('cuda') if device == 'cuda' else None

    best_map3, best_probs, patience = -1.0, None, PATIENCE
    for ep in range(EPOCHS):
        tl = train_one_epoch(model, tr_dl, optim, sched, scaler)
        vprobs, vlabels, vl = eval_loader(model, va_dl)
        v_top1 = float((vprobs.argmax(1) == vlabels).mean())
        v_letters = [OPTIONS[i] for i in vlabels]
        v_map3 = map3_score(scores_to_top3(vprobs), v_letters)
        print(f'  fold {fold_idx} ep {ep+1}: train {tl:.4f}  val {vl:.4f}  top1 {v_top1:.4f}  map3 {v_map3:.4f}')
        if USE_WANDB:
            wandb.log({
                'epoch': ep,
                f'train_loss/fold_{fold_idx}': tl,
                f'val_loss/fold_{fold_idx}':   vl,
                f'val_top1/fold_{fold_idx}':   v_top1,
                f'val_map3/fold_{fold_idx}':   v_map3,
            })
        if v_map3 > best_map3:
            best_map3, best_probs, patience = v_map3, vprobs.copy(), PATIENCE
        else:
            patience -= 1
            if patience <= 0:
                print(f'  early stop at ep {ep+1}')
                break

    del model, tr_ds, va_ds, tr_dl, va_dl, optim, sched, scaler
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_probs

## Run 5-fold CV

In [61]:
if USE_WANDB:
    wandb.init(entity=WANDB_ENTITY, project=WANDB_PROJECT, name='deberta', reinit=True,
               tags=['model:transformer'],
               config={'model': MODEL_ID, 'max_len': MAX_LEN, 'epochs': EPOCHS,
                       'batch': BATCH, 'accum': ACCUM, 'lr': LR, 'folds': FOLDS})
    wandb.define_metric('epoch')
    for m in ['train_loss', 'val_loss', 'val_top1', 'val_map3']:
        wandb.define_metric(f'{m}/*', step_metric='epoch')

In [62]:
oof_grid = np.zeros((len(train), NUM_CHOICES))
t0 = time.time()
for f in range(FOLDS):
    print(f'==== fold {f} ====')
    tr = train[train['fold'] != f].reset_index(drop=True)
    va = train[train['fold'] == f].reset_index(drop=True)
    val_idx = train.index[train['fold'] == f].tolist()
    oof_grid[val_idx] = train_fold(tr, va, f)

elapsed = time.time() - t0
print(f'training time: {elapsed:.0f}s ({elapsed/60:.1f} min)')

==== fold 0 ====


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight              

  fold 0 ep 1: train 1.5892  val 1.5526  top1 0.2525  map3 0.4429
  fold 0 ep 2: train 1.3785  val 1.4690  top1 0.3650  map3 0.5513
  fold 0 ep 3: train 1.0972  val 1.4345  top1 0.3675  map3 0.5563
  fold 0 ep 4: train 0.9670  val 1.4634  top1 0.3425  map3 0.5354
==== fold 1 ====


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight              

  fold 3 ep 1: train 1.6095  val 1.6051  top1 0.2800  map3 0.4417
  fold 3 ep 2: train 1.5929  val 1.5793  top1 0.3025  map3 0.4617
  fold 3 ep 3: train 1.5140  val 1.5374  top1 0.2800  map3 0.4629
  fold 3 ep 4: train 1.2817  val 1.4875  top1 0.3400  map3 0.5058
==== fold 4 ====


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight              

  fold 4 ep 1: train 1.6087  val 1.6076  top1 0.2400  map3 0.3788
  fold 4 ep 2: train 1.6040  val 1.6105  top1 0.1750  map3 0.3521
  fold 4 ep 3: train 1.6122  val 1.6088  top1 0.1800  map3 0.3562
  early stop at ep 3
training time: 3032s (50.5 min)


## OOF metrics

Question-level `top1_acc` and `MAP@3` are the honest metrics for this task. Binary accuracy on the flattened softmax grid at threshold 0.5 is reported for completeness, but note that softmax probabilities average `1/5 = 0.2` so almost nothing crosses 0.5; the number is not a useful signal.

In [63]:
true_letters = train['answer'].tolist()
true_idx = np.array([OPTIONS.index(a) for a in true_letters])
top3 = scores_to_top3(oof_grid)

y_binary = (np.eye(NUM_CHOICES)[true_idx]).flatten().astype(int)
probs_flat = oof_grid.flatten().clip(1e-7, 1 - 1e-7)
pred_flat = (probs_flat > 0.5).astype(int)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, log_loss
final = {
    'accuracy':  accuracy_score(y_binary, pred_flat),
    'precision': precision_score(y_binary, pred_flat, zero_division=0),
    'recall':    recall_score(y_binary, pred_flat, zero_division=0),
    'f1':        f1_score(y_binary, pred_flat, zero_division=0),
    'log_loss':  log_loss(y_binary, probs_flat),
    'top1_acc':  float((oof_grid.argmax(1) == true_idx).mean()),
    'map3':      map3_score(top3, true_letters),
    'training_time_sec': elapsed,
}
print('DeBERTa OOF metrics:')
for k, v in final.items():
    print(f'  {k}: {v:.4f}')

if USE_WANDB:
    for k, v in final.items():
        wandb.run.summary[k] = v

DeBERTa OOF metrics:
  accuracy: 0.8045
  precision: 0.6230
  recall: 0.0570
  f1: 0.1044
  log_loss: 0.4687
  top1_acc: 0.3285
  map3: 0.5002
  training_time_sec: 3032.1261


## Full-train on all data and predict on test

In [64]:
torch.manual_seed(SEED)
model_full = AutoModelForMultipleChoice.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32).to(device)

full_ds = MCDataset(train, tokenizer, has_answer=True)
full_dl = DataLoader(full_ds, batch_size=BATCH, shuffle=True, num_workers=0)
steps = max((len(full_dl) // ACCUM) * EPOCHS, 1)
optim = torch.optim.AdamW(model_full.parameters(), lr=LR, weight_decay=0.01)
sched = get_linear_schedule_with_warmup(optim, int(0.1 * steps), steps)
scaler = torch.amp.GradScaler('cuda') if device == 'cuda' else None

for ep in range(EPOCHS):
    tl = train_one_epoch(model_full, full_dl, optim, sched, scaler)
    print(f'  full ep {ep+1}: loss {tl:.4f}')

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight              

  full ep 1: loss 1.6067
  full ep 2: loss 1.5559
  full ep 3: loss 1.4358
  full ep 4: loss 1.3114


In [65]:
model_full.eval()
test_ds = MCDataset(test, tokenizer, has_answer=False)
test_dl = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=0)

test_grid = []
with torch.no_grad():
    for batch in test_dl:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model_full(**batch)
        test_grid.append(torch.softmax(out.logits, dim=-1).cpu().numpy())
test_grid = np.concatenate(test_grid, axis=0)
print(f'test predictions: {test_grid.shape}')

del model_full, full_ds, full_dl, test_ds, test_dl, optim, sched, scaler
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

test predictions: (500, 5)


## Save predictions

In [66]:
np.save(PREDICTIONS_DIR / 'oof_deberta.npy', oof_grid)
np.save(PREDICTIONS_DIR / 'test_deberta.npy', test_grid)
with open(PREDICTIONS_DIR / 'deberta_metrics.json', 'w') as fh:
    json.dump(final, fh, indent=2)
print(f'saved oof_deberta.npy: {oof_grid.shape}')
print(f'saved test_deberta.npy: {test_grid.shape}')
if USE_WANDB:
    wandb.finish()

saved oof_deberta.npy: (2000, 5)
saved test_deberta.npy: (500, 5)


epoch,▁▃▆█▁▃▆█▁▃▆█▁▃▆█▁▃▆
train_loss/fold_0,█▆▂▁
train_loss/fold_1,██▆▁
train_loss/fold_2,█▇▃▁
train_loss/fold_3,██▆▁
train_loss/fold_4,▅▁█
val_loss/fold_0,█▃▁▃
val_loss/fold_1,██▄▁
val_loss/fold_2,█▂▁▃
val_loss/fold_3,█▆▄▁
+11,...
